In [98]:
#Use the following before running code
!python -m spacy download en_core_web_sm

21689.76s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


zsh:1: command not found: python


In [99]:
# from sklearn.feature_extraction.text import CountVectorizer
# from sklearn.metrics import f1_score
# from sklearn.model_selection import train_test_split
# from sklearn.naive_bayes import ComplementNB
# from spacy import load
# from csv import reader

# def readFromCSV(path: str):
#     data = None
#     with open(path, 'r', encoding='UTF-8') as f:
#         data = list(reader(f))
#     return data

# def preprocess(sents: list[str]):
#     nlp = load("en_core_web_sm", disable=["ner", "parser"])
#     out = []
#     for sent in sents:
#         sent = ' '.join(sent.split())
#         doc = nlp(sent)
#         out.append(' '.join(token.lemma_ for token in doc if not token.is_punct)) # keep stopwords as 'not' can sometimes be a stopword and ruin our sentiment analysis
#     return out


# def main():
#     # Process labeled data
#     labeledData = readFromCSV('data/labeledData.csv')[1:]
#     unlabeledData = readFromCSV('data/unlabeledData.csv')[1:]

#     labeledEmailMessages = preprocess([line[0] for line in labeledData])
#     # labeledRejectionOutcomes = [0 if line[1] == 'reject' else 1 for line in labeledData]
#     labeledRejectionOutcomes = [line[1] for line in labeledData]
#     unlabeledEmailMessages = preprocess([line[9] for line in unlabeledData])
#     #print(unlabeledEmailMessages)

#     bowVectorizer = CountVectorizer(max_features=5000)

#     X = bowVectorizer.fit_transform(labeledEmailMessages) #[line[0] for line in labeledData] is a list of strings
#     y = labeledRejectionOutcomes

#     # print(X.get_shape())
#     # print(len(y))

#     X_train, X_dev, y_train, y_dev = train_test_split(X, y)

#     model = ComplementNB().fit(X_train, y_train)
   
#     #print("Training F1")
#     y_pred_dev = model.predict(X_dev)
#     f1 = f1_score(y_dev, y_pred_dev, average='micro')
#     print(f"F1 score: {f1:.4f}\n")

#     #print("Test Results")
#     X_test = bowVectorizer.transform(unlabeledEmailMessages)
#     y_pred_test = model.predict(X_test)
#     #print(*[reject + ' ' + message for message, reject in zip(unlabeledEmailMessages, y_pred_test)], sep='\n')

# #if __name__ == '__main__':
#     #main()


In [100]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import joblib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from spacy import load
from csv import reader

def readFromCSV(path: str):
    data = None
    with open(path, 'r', encoding='UTF-8') as f:
        data = list(reader(f))
    return data

def preprocess(sents: list[str], nlp):
    out = []
    for sent in sents:
        sent = ' '.join(sent.split())
        doc = nlp(sent)
        out.append(' '.join(token.lemma_ for token in doc if not token.is_punct)) # keep stopwords as 'not' can sometimes be a stopword and ruin our sentiment analysis
    return out

def main(save_model=False):
    # Process labeled data
    nlp = load("en_core_web_sm", disable=["ner", "parser"])
    labeledData = readFromCSV('data/labeledData.csv')[1:]
    unlabeledData = readFromCSV('data/unlabeledData.csv')[1:]
    labeledEmailMessages = preprocess([line[0] for line in labeledData], nlp)
    labeledRejectionOutcomes = [line[1] for line in labeledData]
    unlabeledEmailMessages = preprocess([f"{line[1] } {line[10]} {line[11]}" for line in unlabeledData], nlp)

    bowVectorizer = CountVectorizer(max_features=5000, ngram_range=(1,2))

    X = bowVectorizer.fit_transform(labeledEmailMessages) #[line[0] for line in labeledData] is a list of strings
    y = labeledRejectionOutcomes

    X_train, X_dev, y_train, y_dev = train_test_split(X, y, random_state=42, stratify=y)

    model = ComplementNB().fit(X_train, y_train)

    if save_model:
        joblib.dump(model, 'model.joblib')
        joblib.dump(bowVectorizer, 'vectorizer.joblib')
   
    #print("Training F1")
    y_pred_dev = model.predict(X_dev)
    f1 = f1_score(y_dev, y_pred_dev, average='macro')
    print(f"F1 score: {f1:.4f}\n")


    print("Test Results")
    X_test = bowVectorizer.transform(unlabeledEmailMessages)
    y_pred_test = model.predict(X_test)
    print(*[f"{pred}: {row[10]} - {row[11]}" for row, pred in zip(unlabeledData, y_pred_test)], sep='\n')
    
    print("\nClas. Report:", classification_report(y_dev, y_pred_dev))
    print("\nConfusion Matrix: ", confusion_matrix(y_dev, y_pred_dev))
    print("\nAccuracy Score: ", f"{accuracy_score(y_dev, y_pred_dev):.2f}")

if __name__ == '__main__':
    main(save_model=True)

# run with save_model=True to save the model and vectorizer for later use
# run with save_model=False to just evaluate the model without saving

F1 score: 0.7746

Test Results
reject: Tesco - Decision Scientist at Tesco: we’ve got your application
reject: Healthify - Michael Gary Scott, your application was sent to Healthify
reject: Amazon - Thank you for Applying to Amazon!
not_reject: IBM - You have successfully submitted your IBM job application - 14499 - Business Analyst-ADM
reject: IBM - Your IBM Application: Next Steps
reject: Infineon - Welcome to Infineon Technologies Careers
reject: Generative Futures - Michael Gary Scott, your application was sent to Generative Futures
reject: Coursera - Thank you for applying at Coursera
reject: Coursera - Thank you for applying at Coursera
reject: Xylem - Thank you for your interest in Xylem! Your application has been received.
not_reject: Amazon - Action Required for your Amazon Application – Complete Assessment
not_reject: Amazon - Keep track of your application
reject: Awign - Michael Gary Scott, your application was sent to Awign
reject: Bhanzu - Application for Data Analyst rec

In [101]:
# #hugging face
# !pip install -U huggingface_hub 

# from huggingface_hub import HfApi

# api = HfApi(token="token goes here")

# api.create_repo("rhallett/COS498-Final-Project", exist_ok=True)

# api.upload_file(path_or_fileobj="model.joblib", path_in_repo="model.joblib", repo_id="rhallett/COS498-Final-Project")
# api.upload_file(path_or_fileobj="vectorizer.joblib", path_in_repo="vectorizer.joblib", repo_id="rhallett/COS498-Final-Project")

https://huggingface.co/rhallett/COS498-Final-Project/tree/main 

In [ ]:
from huggingface_hub import hf_hub_download
nlp = load("en_core_web_sm", disable=["ner", "parser"])

model = joblib.load(hf_hub_download(repo_id="rhallett/COS498-ANLP-Final-Project", filename="model.joblib"))
vectorizer = joblib.load(hf_hub_download(repo_id="rhallett/COS498-ANLP-Final-Project", filename="vectorizer.joblib"))

# Real emails i've recieved! 
emails_list = [
"""Dear Ryan Hallett (CAND-348984), I hope you're doing well. Thank you for applying to the 2026 Summer Data Analytics/Data Science Intern role at Avangrid. After reviewing your application, we’ve decided to move forward with other candidates. We appreciate your interest and encourage you to apply for future opportunities that match your skills. Best regards, Avangrid Early Careers""",
"""Good morning, thank you for applying to the Cianbro IT internship on our website. We would like to set up a virtual call for an interview. Would you have availability on Monday, December 10th, @ 2:00 pm? Thank you, Chrissy.""",
"""Thank you for taking the time to submit your recent application for our Embedded Software Engineer Intern opening. Our team was able to review your application and we value your qualifications and background. However, after careful consideration we have decided to move forward with other candidates whose skills more closely aligned to our current and future needs. We appreciate your interest in Bose and know it’s discouraging to hear undesirable news, but we are always opening new roles and would love to consider you in the future. Please continue to visit our career site, and we hope to stay connected. Thank you again, Bose Talent Acquisition"""
]

for email in emails_list:
    email_processing = preprocess([email], nlp)
    email_vec = vectorizer.transform(email_processing)  
    email_prediction = model.predict(email_vec)[0]
    print(f"this email is predicted to be: {email_prediction}")

print("IDEAL OUTPUT: reject, not_reject, reject")

this email is predicted to be: reject
this email is predicted to be: not_reject
this email is predicted to be: reject
IDEAL OUTPUT: reject, not_reject, reject
